# Gas Fee Prediction — Comprehensive Model Analysis

### XGBoost · Random Forest · CatBoost · LightGBM

**Full pipeline**: Data → EDA → Feature Engineering → Training → Evaluation → Overfitting → Outliers → Metrics → Comparison

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import os

from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb

warnings.filterwarnings('ignore')

# Professional Light-Mode Styling
plt.rcParams.update({
    'figure.facecolor': '#ffffff', 'axes.facecolor': '#f8f9fa',
    'axes.edgecolor': '#dee2e6', 'axes.labelcolor': '#212529',
    'axes.titleweight': 'bold', 'axes.grid': True,
    'grid.color': '#e9ecef', 'grid.linestyle': '--', 'grid.alpha': 0.7,
    'xtick.color': '#495057', 'ytick.color': '#495057',
    'text.color': '#212529', 'font.family': 'sans-serif',
    'font.size': 11, 'figure.dpi': 120, 'figure.figsize': (14, 6),
})

MODEL_COLORS = {'XGBoost': '#e67e22', 'RandomForest': '#27ae60',
                'CatBoost': '#2980b9', 'LightGBM': '#8e44ad'}

print('✅ All imports loaded successfully.')

---
## 1. Data Loading & Simulation

Simulating realistic Ethereum data with EIP-1559 mechanics (5000 blocks).

In [ ]:
def simulate_blockchain_data(n_blocks=5000, seed=42):
    np.random.seed(seed)
    start = datetime(2024, 3, 1)
    ts = [start + timedelta(seconds=12*i) for i in range(n_blocks)]
    bn = np.arange(19_000_000, 19_000_000 + n_blocks)
    gl = np.random.normal(30e6, 5e5, n_blocks).clip(15e6, 36e6)
    hod = np.array([t.hour for t in ts])
    txb = 150 + 80 * np.sin((hod - 9) * np.pi / 12)
    tc = np.random.normal(txb, 30, n_blocks).clip(10, 300).astype(int)
    gur = ((tc / 300) * np.random.normal(0.85, 0.1, n_blocks)).clip(0.1, 1.0)
    gu = (gur * gl).astype(int)
    bf = np.zeros(n_blocks); bf[0] = 30e9
    for i in range(1, n_blocks):
        u = gu[i-1] / gl[i-1]
        bf[i] = np.clip(bf[i-1] * (1 + 0.125 * ((u - 0.5) / 0.5)), 1e9, 500e9)
    mp = gu / gl
    pf = (np.random.lognormal(1.5, 0.8, n_blocks) * 1e9 * mp).clip(1e9, 50e9)
    ms = (tc * np.random.uniform(2, 8, n_blocks)).astype(int)
    gp = (bf + pf) / 1e9
    df = pd.DataFrame({'block_number': bn, 'timestamp': ts, 'gas_used': gu,
        'gas_limit': gl, 'base_fee': bf, 'transaction_count': tc,
        'mempool_size': ms, 'priority_fee': pf, 'gas_price_gwei': gp})
    for col in ['mempool_size', 'priority_fee']:
        df.loc[np.random.rand(n_blocks) < 0.03, col] = np.nan
    return df

df_raw = simulate_blockchain_data(5000)
print(f'Dataset: {df_raw.shape}  |  Time: {df_raw["timestamp"].min()} → {df_raw["timestamp"].max()}')
print(f'\nMissing:\n{df_raw.isna().sum()[df_raw.isna().sum() > 0]}')
df_raw.head()

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
df_raw.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(df_raw['timestamp'], df_raw['gas_price_gwei'], lw=0.5, color='#2980b9', alpha=0.8)
axes[0].set_title('Gas Price Over Time'); axes[0].set_ylabel('Gwei')
axes[1].hist(df_raw['gas_price_gwei'], bins=60, color='#27ae60', edgecolor='white', alpha=0.85)
axes[1].axvline(df_raw['gas_price_gwei'].mean(), color='#c0392b', ls='--', lw=2, label=f'Mean={df_raw["gas_price_gwei"].mean():.1f}')
axes[1].set_title('Gas Price Distribution'); axes[1].legend()
axes[2].boxplot([df_raw['gas_price_gwei'].dropna()], labels=['gas_price'],
                patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.6))
axes[2].set_title('Box Plot')
plt.tight_layout(); plt.show()

In [ ]:
corr = df_raw.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax, square=True, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
df_raw['hour'] = df_raw['timestamp'].dt.hour
hourly = df_raw.groupby('hour')['gas_price_gwei'].agg(['mean', 'std'])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly.index, hourly['mean'], 'o-', color='#e67e22', lw=2, ms=6)
ax.fill_between(hourly.index, hourly['mean']-hourly['std'], hourly['mean']+hourly['std'], alpha=0.15, color='#e67e22')
ax.set_title('Average Gas Price by Hour (UTC)'); ax.set_xlabel('Hour'); ax.set_ylabel('Gwei'); ax.set_xticks(range(24))
plt.tight_layout(); plt.show()
df_raw.drop(columns=['hour'], inplace=True)

---
## 3. Feature Engineering

Creating **35 ML-ready features**: temporal, utilization, rolling, lag, EIP-1559 momentum, mempool.

In [ ]:
def engineer_features(df):
    df = df.copy().sort_values('block_number').reset_index(drop=True)
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['month'] = df['timestamp'].dt.month
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['dow_sin'] = np.sin(2*np.pi*df['day_of_week']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['day_of_week']/7)
    df['block_utilization'] = df['gas_used'] / df['gas_limit']
    df['tx_density'] = df['transaction_count'] / df['gas_limit'] * 1e6
    for w in [5, 15, 50]:
        df[f'base_fee_roll_{w}'] = df['base_fee'].shift(1).rolling(w, min_periods=1).mean()
        df[f'utilization_roll_{w}'] = df['block_utilization'].shift(1).rolling(w, min_periods=1).mean()
        df[f'tx_count_roll_{w}'] = df['transaction_count'].shift(1).rolling(w, min_periods=1).mean()
    for lag in [1, 2, 3, 5, 10]:
        df[f'gas_price_lag_{lag}'] = df['gas_price_gwei'].shift(lag)
        df[f'base_fee_lag_{lag}'] = df['base_fee'].shift(lag)
    df['base_fee_pct_change'] = df['base_fee'].pct_change(1).replace([np.inf, -np.inf], 0)
    df['base_fee_acceleration'] = df['base_fee_pct_change'].diff(1)
    df['mempool_size'] = df['mempool_size'].fillna(df['mempool_size'].rolling(10, min_periods=1).median()).fillna(500)
    df['priority_fee'] = df['priority_fee'].fillna(df['priority_fee'].rolling(10, min_periods=1).median()).fillna(2e9)
    df['mempool_pressure'] = df['mempool_size'] / (df['transaction_count'] + 1)
    df['priority_fee_gwei'] = df['priority_fee'] / 1e9
    df['base_fee_gwei'] = df['base_fee'] / 1e9
    return df

df_feat = engineer_features(df_raw)
print(f'Features engineered: {df_feat.shape[1]} columns')

---
## 4. Preprocessing & Train/Test Split

> ⚠️ Chronological split — never random shuffle for time-series.

In [ ]:
drop_cols = ['block_number', 'timestamp', 'gas_price_gwei', 'base_fee', 'priority_fee', 'hour', 'day_of_week', 'month']
feature_cols = [c for c in df_feat.columns if c not in drop_cols]
lr = [c for c in df_feat.columns if any(t in c for t in ['_lag_', '_roll_', '_pct_change', '_acceleration'])]
df_feat[lr] = df_feat[lr].ffill().bfill().fillna(0)
df_feat = df_feat.dropna(subset=['gas_price_gwei']).reset_index(drop=True)
X = df_feat[feature_cols].fillna(0).copy()
y = df_feat['gas_price_gwei'].copy()
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  |  Features: {X_train.shape[1]}')

---
## 5. Outlier Detection & Analysis

IQR method and Z-score outlier identification.

In [ ]:
Q1, Q3 = y.quantile(0.25), y.quantile(0.75)
IQR = Q3 - Q1
lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
iqr_out = ((y < lo) | (y > hi)).sum()
zs = np.abs((y - y.mean()) / y.std())
z_out = (zs > 3).sum()
print(f'IQR: Q1={Q1:.2f}, Q3={Q3:.2f}, Bounds=[{lo:.2f}, {hi:.2f}], Outliers={iqr_out} ({iqr_out/len(y)*100:.1f}%)')
print(f'Z-Score (|z|>3): Outliers={z_out} ({z_out/len(y)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bp = axes[0].boxplot([y_train, y_test], labels=['Train', 'Test'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2980b9'); bp['boxes'][1].set_facecolor('#e67e22')
axes[0].set_title('Train vs Test Distribution'); axes[0].set_ylabel('Gwei')
axes[1].hist(zs, bins=50, color='#27ae60', edgecolor='white', alpha=0.8)
axes[1].axvline(3, color='#c0392b', ls='--', lw=2, label='|z|=3')
axes[1].set_title('Z-Score Distribution'); axes[1].legend()
is_out = (y < lo) | (y > hi)
axes[2].scatter(range(len(y[~is_out])), y[~is_out], s=2, alpha=0.3, color='#2980b9', label='Normal')
axes[2].scatter(np.where(is_out)[0], y[is_out], s=10, color='#c0392b', label='Outlier', zorder=5)
axes[2].set_title('Outlier Identification'); axes[2].legend()
plt.tight_layout(); plt.show()

---
## 6. Model Training — All Four Models

In [ ]:
models, preds_train, preds_test, timings = {}, {}, {}, {}
vs = int(len(X_train) * 0.9)

# XGBoost
print('Training XGBoost...'); t0 = time.time()
m = XGBRegressor(n_estimators=1000, max_depth=5, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, gamma=0.1,
    objective='reg:squarederror', tree_method='hist', random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='rmse')
m.fit(X_train.iloc[:vs], y_train.iloc[:vs], eval_set=[(X_train.iloc[vs:], y_train.iloc[vs:])], verbose=False)
models['XGBoost'] = m; timings['XGBoost'] = time.time()-t0
print(f'  Done in {timings["XGBoost"]:.2f}s, best iter={m.best_iteration}')

# Random Forest
print('Training RandomForest...'); t0 = time.time()
m = RandomForestRegressor(n_estimators=300, max_depth=20, max_features='sqrt',
    min_samples_leaf=2, bootstrap=True, oob_score=True, max_samples=0.8, random_state=42, n_jobs=-1)
m.fit(X_train, y_train)
models['RandomForest'] = m; timings['RandomForest'] = time.time()-t0
print(f'  Done in {timings["RandomForest"]:.2f}s, OOB R²={m.oob_score_:.4f}')

# CatBoost
print('Training CatBoost...'); t0 = time.time()
m = CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.05, l2_leaf_reg=5,
    early_stopping_rounds=50, random_seed=42, verbose=0, allow_writing_files=False)
m.fit(Pool(X_train.iloc[:vs], y_train.iloc[:vs]), eval_set=Pool(X_train.iloc[vs:], y_train.iloc[vs:]))
models['CatBoost'] = m; timings['CatBoost'] = time.time()-t0
print(f'  Done in {timings["CatBoost"]:.2f}s, best iter={m.get_best_iteration()}')

# LightGBM
print('Training LightGBM...'); t0 = time.time()
m = lgb.LGBMRegressor(n_estimators=2000, num_leaves=63, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.1,
    reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)
m.fit(X_train.iloc[:vs], y_train.iloc[:vs], eval_set=[(X_train.iloc[vs:], y_train.iloc[vs:])],
      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
models['LightGBM'] = m; timings['LightGBM'] = time.time()-t0
print(f'  Done in {timings["LightGBM"]:.2f}s, best iter={m.best_iteration_}')

for n, mdl in models.items():
    preds_train[n] = np.clip(mdl.predict(X_train), 0, None)
    preds_test[n] = np.clip(mdl.predict(X_test), 0, None)
print('\n✅ All 4 models trained + predictions generated!')

---
## 7. Regression Metrics — MAE, MSE, RMSE, R², MAPE

In [ ]:
def reg_metrics(yt, yp):
    return {'MAE': mean_absolute_error(yt,yp), 'MSE': mean_squared_error(yt,yp),
            'RMSE': np.sqrt(mean_squared_error(yt,yp)), 'R²': r2_score(yt,yp),
            'MAPE': np.mean(np.abs((yt-yp)/(yt+1e-8)))*100}

metrics_train = {n: reg_metrics(y_train, preds_train[n]) for n in models}
metrics_test = {n: reg_metrics(y_test, preds_test[n]) for n in models}
df_mt = pd.DataFrame(metrics_test).T
df_mtr = pd.DataFrame(metrics_train).T

print('TEST SET METRICS:'); display(df_mt.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', subset=['MAE','MSE','RMSE','MAPE']).background_gradient(cmap='RdYlGn', subset=['R²']))
print('\nTRAIN SET METRICS:'); display(df_mtr.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', subset=['MAE','MSE','RMSE','MAPE']).background_gradient(cmap='RdYlGn', subset=['R²']))

---
## 8. Overfitting Analysis

**Key indicator**: Large gap between train and test R² = overfitting.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
mnames = list(models.keys()); x = np.arange(len(mnames)); w = 0.35
for ax, met in zip(axes, ['MAE', 'RMSE', 'R²', 'MAPE']):
    tv = [df_mtr.loc[m, met] for m in mnames]
    ev = [df_mt.loc[m, met] for m in mnames]
    b1 = ax.bar(x-w/2, tv, w, label='Train', color='#3498db', alpha=0.8, edgecolor='white')
    b2 = ax.bar(x+w/2, ev, w, label='Test', color='#e74c3c', alpha=0.8, edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(mnames, rotation=15, ha='right', fontsize=9)
    ax.set_title(met, fontweight='bold'); ax.legend(fontsize=8)
    for bar in list(b1)+list(b2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
fig.suptitle('Train vs Test — Overfitting Check', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

overfit = pd.DataFrame(index=mnames, columns=['Train R²', 'Test R²', 'Gap', 'Overfit?'])
for m in mnames:
    tr, te = df_mtr.loc[m,'R²'], df_mt.loc[m,'R²']
    overfit.loc[m] = [f'{tr:.4f}', f'{te:.4f}', f'{tr-te:.4f}', '⚠️ Yes' if tr-te > 0.05 else '✅ No']
print('\nOverfitting Summary:'); display(overfit)

---
## 9. Classification Metrics — Accuracy, Precision, Recall, F1

Gas prices binned into **Low** (<Q1) / **Medium** (Q1–Q3) / **High** (>Q3) for classification metrics.

In [ ]:
q25, q75 = y.quantile(0.25), y.quantile(0.75)
def to_cat(v): return pd.cut(v, bins=[-np.inf, q25, q75, np.inf], labels=['Low','Medium','High'])
y_test_cat = to_cat(y_test)

cls_results = {}
for name in models:
    ypc = to_cat(preds_test[name])
    cls_results[name] = {
        'Accuracy': accuracy_score(y_test_cat, ypc),
        'Precision(wt)': precision_score(y_test_cat, ypc, average='weighted', zero_division=0),
        'Recall(wt)': recall_score(y_test_cat, ypc, average='weighted', zero_division=0),
        'F1(wt)': f1_score(y_test_cat, ypc, average='weighted', zero_division=0),
        'Precision(macro)': precision_score(y_test_cat, ypc, average='macro', zero_division=0),
        'Recall(macro)': recall_score(y_test_cat, ypc, average='macro', zero_division=0),
        'F1(macro)': f1_score(y_test_cat, ypc, average='macro', zero_division=0),
    }

display(pd.DataFrame(cls_results).T.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
labels = ['Low', 'Medium', 'High']
for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test_cat, to_cat(preds_test[name]), labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=labels, yticklabels=labels, linewidths=0.5)
    ax.set_title(f'{name}', fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

best = max(cls_results, key=lambda n: cls_results[n]['F1(wt)'])
print(f'\n📊 Classification Report — Best: {best}')
print(classification_report(y_test_cat, to_cat(preds_test[best]), target_names=labels, zero_division=0))

---
## 10. Visual Model Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
n = min(500, len(y_test)); ix = np.arange(n)
for ax, (name, color) in zip(axes.flat, MODEL_COLORS.items()):
    ax.plot(ix, y_test.values[:n], lw=0.8, color='#2c3e50', alpha=0.7, label='Actual')
    ax.plot(ix, preds_test[name][:n], lw=0.8, color=color, ls='--', alpha=0.9, label=name)
    ax.fill_between(ix, y_test.values[:n], preds_test[name][:n], alpha=0.08, color=color)
    m = metrics_test[name]
    ax.set_title(f'{name} — RMSE:{m["RMSE"]:.3f} R²:{m["R²"]:.4f} MAPE:{m["MAPE"]:.2f}%', fontweight='bold', fontsize=11)
    ax.set_xlabel('Block Index'); ax.set_ylabel('Gwei'); ax.legend(fontsize=9)
fig.suptitle('Actual vs Predicted — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (name, color) in zip(axes, MODEL_COLORS.items()):
    ax.scatter(y_test, preds_test[name], s=5, alpha=0.2, color=color)
    mv = max(y_test.max(), preds_test[name].max())
    ax.plot([0,mv],[0,mv], 'k--', lw=1, alpha=0.5, label='Perfect')
    ax.set_title(f'{name} (R²={metrics_test[name]["R²"]:.4f})', fontweight='bold')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted'); ax.legend(fontsize=8)
fig.suptitle('Scatter: Actual vs Predicted', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (name, color) in zip(axes, MODEL_COLORS.items()):
    res = y_test.values - preds_test[name]
    ax.hist(res, bins=50, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(0, color='#c0392b', ls='--', lw=2)
    ax.set_title(f'{name}\nMean={np.mean(res):.3f} Std={np.std(res):.3f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Residual (Gwei)')
fig.suptitle('Residual Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(timings.keys(), timings.values(), color=[MODEL_COLORS[m] for m in timings], alpha=0.85, edgecolor='white', width=0.5)
for b, v in zip(bars, timings.values()): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.2f}s', ha='center', va='bottom', fontweight='bold')
ax.set_title('Training Time', fontweight='bold', fontsize=14); ax.set_ylabel('Seconds')
plt.tight_layout(); plt.show()

---
## 11. TimeSeriesSplit Cross-Validation (5-fold)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv = {n: {'rmse':[], 'mae':[], 'r2':[]} for n in models}
for fold, (tri, vli) in enumerate(tscv.split(X_train), 1):
    Xtr, ytr = X_train.iloc[tri], y_train.iloc[tri]
    Xvl, yvl = X_train.iloc[vli], y_train.iloc[vli]
    print(f'Fold {fold}: Train={len(Xtr):,} Val={len(Xvl):,}')
    for name, Cls in [('XGBoost', XGBRegressor), ('RandomForest', RandomForestRegressor),
                       ('CatBoost', CatBoostRegressor), ('LightGBM', lgb.LGBMRegressor)]:
        if name=='XGBoost': fm = Cls(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1); fm.fit(Xtr, ytr)
        elif name=='RandomForest': fm = Cls(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1); fm.fit(Xtr, ytr)
        elif name=='CatBoost': fm = Cls(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0, allow_writing_files=False); fm.fit(Xtr, ytr)
        else: fm = Cls(n_estimators=300, num_leaves=63, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1); fm.fit(Xtr, ytr)
        yp = fm.predict(Xvl)
        cv[name]['rmse'].append(np.sqrt(mean_squared_error(yvl,yp)))
        cv[name]['mae'].append(mean_absolute_error(yvl,yp))
        cv[name]['r2'].append(r2_score(yvl,yp))
print('\n✅ CV complete!')

In [ ]:
cv_df = pd.DataFrame({n: {f'{m} (mean±std)': f'{np.mean(cv[n][m.lower()]):.4f} ± {np.std(cv[n][m.lower()]):.4f}' for m in ['RMSE','MAE','R²']} for n in models}).T
display(cv_df)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(5); w = 0.2
for i, (name, color) in enumerate(MODEL_COLORS.items()):
    ax.bar(x + i*w, cv[name]['rmse'], w, label=name, color=color, alpha=0.85, edgecolor='white')
ax.set_xticks(x+1.5*w); ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_title('CV RMSE per Fold', fontweight='bold', fontsize=14); ax.set_ylabel('RMSE (Gwei)'); ax.legend()
plt.tight_layout(); plt.show()

---
## 12. Feature Importance — All Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16)); top = 15
fi = pd.Series(models['XGBoost'].feature_importances_, index=feature_cols).nlargest(top)
axes[0,0].barh(fi.index, fi.values, color='#e67e22', alpha=0.85)
axes[0,0].set_title('XGBoost', fontweight='bold'); axes[0,0].invert_yaxis()
fi = pd.Series(models['RandomForest'].feature_importances_, index=feature_cols).nlargest(top)
axes[0,1].barh(fi.index, fi.values, color='#27ae60', alpha=0.85)
axes[0,1].set_title('Random Forest (MDI)', fontweight='bold'); axes[0,1].invert_yaxis()
fi = pd.Series(models['CatBoost'].get_feature_importance(), index=feature_cols).nlargest(top)
axes[1,0].barh(fi.index, fi.values, color='#2980b9', alpha=0.85)
axes[1,0].set_title('CatBoost', fontweight='bold'); axes[1,0].invert_yaxis()
fi = pd.Series(models['LightGBM'].booster_.feature_importance('gain'), index=feature_cols).nlargest(top)
axes[1,1].barh(fi.index, fi.values, color='#8e44ad', alpha=0.85)
axes[1,1].set_title('LightGBM (Gain)', fontweight='bold'); axes[1,1].invert_yaxis()
fig.suptitle('Top 15 Feature Importances', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 13. Final Summary & Winner

In [ ]:
summary = pd.DataFrame(index=list(models.keys()))
for m in models:
    summary.loc[m,'RMSE'] = metrics_test[m]['RMSE']
    summary.loc[m,'MAE'] = metrics_test[m]['MAE']
    summary.loc[m,'MSE'] = metrics_test[m]['MSE']
    summary.loc[m,'R²'] = metrics_test[m]['R²']
    summary.loc[m,'MAPE(%)'] = metrics_test[m]['MAPE']
    summary.loc[m,'F1(wt)'] = cls_results[m]['F1(wt)']
    summary.loc[m,'Accuracy'] = cls_results[m]['Accuracy']
    summary.loc[m,'Precision'] = cls_results[m]['Precision(wt)']
    summary.loc[m,'Recall'] = cls_results[m]['Recall(wt)']
    summary.loc[m,'Train(s)'] = timings[m]
    summary.loc[m,'CV_R²'] = np.mean(cv[m]['r2'])

display(summary.astype(float).style.format('{:.4f}')
    .background_gradient(cmap='RdYlGn_r', subset=['RMSE','MAE','MSE','MAPE(%)','Train(s)'])
    .background_gradient(cmap='RdYlGn', subset=['R²','F1(wt)','Accuracy','Precision','Recall','CV_R²']))

winner = summary['R²'].astype(float).idxmax()
print(f'\n🏆 WINNER (by Test R²): {winner}')
print(f'   R²={summary.loc[winner,"R²"]}  RMSE={summary.loc[winner,"RMSE"]}  F1={summary.loc[winner,"F1(wt)"]}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
mets = ['R²', 'F1(wt)', 'Accuracy', 'Precision', 'Recall']
x = np.arange(len(mets)); w = 0.2
for i, (name, color) in enumerate(MODEL_COLORS.items()):
    vals = [float(summary.loc[name, m]) for m in mets]
    bars = ax.bar(x+i*w, vals, w, label=name, color=color, alpha=0.85, edgecolor='white')
    for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
ax.set_xticks(x+1.5*w); ax.set_xticklabels(mets, fontsize=11)
ax.set_title('Final Model Comparison', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.15); ax.legend()
plt.tight_layout(); plt.show()
print('\n✅ Analysis complete!')

---
### 📋 Summary
| Section | Contents |
|---|---|
| 1 | Data simulation (5000 blocks, EIP-1559) |
| 2 | EDA — distributions, correlations, hourly |
| 3 | Feature engineering — 35 features |
| 4 | Preprocessing — chronological 80/20 split |
| 5 | Outlier detection — IQR + Z-Score |
| 6 | Model training — XGBoost, RF, CatBoost, LightGBM |
| 7 | Regression metrics — MAE, MSE, RMSE, R², MAPE |
| 8 | Overfitting analysis — train vs test gap |
| 9 | Classification metrics — Accuracy, Precision, Recall, F1 |
| 10 | Visual comparison — time series, scatter, residuals |
| 11 | Cross-validation — 5-fold TimeSeriesSplit |
| 12 | Feature importance — all 4 models |
| 13 | Final summary & winner |